<a href="https://colab.research.google.com/github/Sega757/chronos-completion-authority-separated-agent-harn/blob/main/chronos_completion_authority_separated_agent_harn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chronos: Completion-Authority-Separated Agent Harness (Gemma 4 Developer Agent)

**Idea:** the model never decides that a task is solved. It only *proposes* a patch; the physical evaluator returns the verdict, and every claim/proof pair is written to a tamper-evident local log.

**What is mine vs. provided by the organizers**

| Part | Origin |
|---|---|
| Env fix (cutlass hooks, `+cu128` wheel names), vLLM server, `Evaluator`, `swegemma.*`, zip limits | organizers' infrastructure (required to run in the offline environment) |
| `transponder.py` (atomic JSONL bus, flock+fsync, 9-event schema) | **own** |
| `audit.py` (hash chain, passive observer with 0/1/2 exit codes) | **own** |
| Claim-vs-proof loop, divergence metric, optional C-T submission gate | **own** |
| Agent logic inside the submission dir (`agent.yaml`, prompts, tools) | see section 3, edit here |

Honest limits: hash chain uses SHA-256 (BLAKE3/Ed25519 signing is roadmap); canonical JSON is JCS-like, not full RFC 8785.

## 1. Environment (organizer-required fixes)

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path

ENV = {
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1',
    'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200', 'VLLM_NO_USAGE_STATS': '1',
    'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}
os.environ.update(ENV)

for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
            '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for p in glob.glob(pat):
        try: os.unlink(p)
        except OSError: pass

WHEELHOUSE = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp / name).exists(): os.symlink(w, tmp / name)

wheels = sorted(map(str, tmp.glob('*.whl')))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print(f'installed {len(wheels)} wheels')

installed 41 wheels


## 2. Own modules: telemetry bus and audit ledger

In [ ]:
from pathlib import Path
Path('/kaggle/working/chronos').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/telemetry').mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /kaggle/working/chronos/transponder.py
"""META-CORE transponder: local atomic JSONL event bus (flock + fsync). No network, no git."""
import fcntl, json, os, uuid
from datetime import datetime, timezone
from pathlib import Path

EVENTS = ("CORE_STARTED", "CORE_STOPPED", "HEARTBEAT", "STEP_STARTED", "STEP_COMPLETED",
          "STEP_FAILED", "SPAWN_REQUESTED", "SPAWN_COMPLETED", "FATAL_ERROR")
REQUIRED = ("schema", "event_id", "core_id", "event", "timestamp")
TRACED = ("STEP_", "SPAWN_")  # ADR-002: trace ids mandatory for execution events

LOG_PATH = Path(os.environ.get("METACORE_LOG", "/kaggle/working/telemetry/event_stream.jsonl"))


def _canon(obj):
    # JCS-like canonical form (sorted keys, no whitespace). Not a full RFC 8785 implementation.
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def emit(event, core_id, log_path=None, **fields):
    if event not in EVENTS:
        raise ValueError(f"unknown event {event!r}")
    rec = {"schema": "v1.0", "event_id": str(uuid.uuid4()), "core_id": core_id,
           "event": event, "timestamp": datetime.now(timezone.utc).isoformat()}
    rec.update(fields)
    if event.startswith(TRACED):
        rec.setdefault("trace_id", uuid.uuid4().hex)
        rec.setdefault("span_id", uuid.uuid4().hex[:16])
    missing = [k for k in REQUIRED if k not in rec]
    if missing:
        raise ValueError(f"missing fields: {missing}")
    path = Path(log_path or LOG_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    line = _canon(rec) + "\n"
    with open(path, "a", encoding="utf-8") as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        try:
            f.write(line)
            f.flush()
            os.fsync(f.fileno())
        finally:
            fcntl.flock(f, fcntl.LOCK_UN)
    return rec


Writing /kaggle/working/chronos/transponder.py


In [ ]:
%%writefile /kaggle/working/chronos/audit.py
"""Hash-chain ledger + passive observer over event_stream.jsonl (read-only)."""
import hashlib, json
from pathlib import Path


def _canon(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def read_events(log_path):
    p = Path(log_path)
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]


def build_chain(events, out_path):
    # H_t = SHA256(H_{t-1} || canonical(event)). Stand-in for BLAKE3+Ed25519 (roadmap).
    prev, rows = "0" * 64, []
    for e in events:
        h = hashlib.sha256((prev + _canon(e)).encode()).hexdigest()
        rows.append({"hash": h, "prev": prev, "event_id": e["event_id"]})
        prev = h
    Path(out_path).write_text("\n".join(_canon(r) for r in rows) + "\n", encoding="utf-8")
    return prev


def verify_chain(events, chain_path):
    rows = [json.loads(l) for l in Path(chain_path).read_text().splitlines() if l.strip()]
    if len(rows) != len(events):
        return False, f"length mismatch {len(rows)} vs {len(events)}"
    prev = "0" * 64
    for i, (e, r) in enumerate(zip(events, rows)):
        h = hashlib.sha256((prev + _canon(e)).encode()).hexdigest()
        if r["prev"] != prev or r["hash"] != h:
            return False, f"tamper detected at event #{i}"
        prev = h
    return True, f"ok, head={prev[:12]}"


def observe(events, max_depth=4):
    # Exit-code contract: 0 OK, 1 WARN, 2 CRITICAL.
    open_spans, parents, issues, level = {}, {}, [], 0
    for e in events:
        sid = e.get("span_id")
        if e["event"] == "STEP_STARTED":
            open_spans[sid] = e
            parents[sid] = e.get("parent_span_id")
        elif e["event"] in ("STEP_COMPLETED", "STEP_FAILED"):
            open_spans.pop(sid, None)
        elif e["event"] == "FATAL_ERROR":
            issues.append("FATAL_ERROR logged"); level = 2
    for sid in open_spans:
        issues.append(f"unclosed span {sid}"); level = max(level, 1)
    for sid in parents:
        d, cur, seen = 0, sid, set()
        while parents.get(cur):
            if cur in seen:
                issues.append(f"cycle at {sid}"); level = 2; break
            seen.add(cur); cur = parents[cur]; d += 1
        if d > max_depth:
            issues.append(f"depth {d} at {sid}"); level = max(level, 1)
    return level, issues


Writing /kaggle/working/chronos/audit.py


In [ ]:
sys.path.insert(0, '/kaggle/working/chronos')
import transponder as T, audit as A
LOG = Path('/kaggle/working/telemetry/event_stream.jsonl')
LOG.unlink(missing_ok=True)
T.LOG_PATH = LOG
CORE = 'chronos-core-001'
T.emit('CORE_STARTED', CORE)

{'schema': 'v1.0',
 'event_id': 'cbc999ff-0cda-4572-8046-0e8d74bac1c5',
 'core_id': 'chronos-core-001',
 'event': 'CORE_STARTED',
 'timestamp': '2026-10-04T15:21:22.319565+00:00'}

## 3. Data, submission directory, sampling config

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists(): shutil.rmtree(AGENT_DIR)
shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR)

# thinking_level must NOT be set (LiteLLM maps it to reasoning_effort -> payload errors)
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
    'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 16384\n'
    'thinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', encoding='utf-8')

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
print(len(tasks), 'tasks')

# Where your own agent logic goes: inspect what the starter ships, then edit these files.
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file(): print(p.relative_to(AGENT_DIR), p.stat().st_size, 'bytes')

129 tasks
adapters/main_lora/adapter_config.json 664 bytes
adapters/main_lora/adapter_model.safetensors 217672 bytes
adapters/tool_lora/adapter_config.json 664 bytes
adapters/tool_lora/adapter_model.safetensors 217672 bytes
agent.yaml 438 bytes
configs/sampling.yaml 120 bytes
eval_config.yaml 232 bytes
prompts/analyzer.md 527 bytes
prompts/system.md 3888 bytes
sub_agents/code_analyzer.yaml 361 bytes


## 4. Graph tools smoke test (organizer API)

In [ ]:
from swegemma import graph as sg
GRAPH_DIR, EMBEDDINGS_DIR = str(DATA_DIR / 'graphs'), str(DATA_DIR / 'embeddings')
t0 = tasks[0]
g = sg.get_graph(repo_name=t0.repo, graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, base_commit=t0.base_commit)
node = next((n for n, d in g.degree() if d >= 5), next(iter(g.nodes())))
nb = sg.get_neighbor(node=node, graph=g, max_neighbors=10)
sub = sg.get_induced_subgraph(g, [node, *nb[:4]])
print(t0.repo, g.number_of_nodes(), 'nodes ->', sub.number_of_nodes(), 'in focal subgraph')

fastapi/fastapi 4263 nodes -> 4 in focal subgraph


## 5. vLLM server

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')

declared = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
n = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp = 4 if n >= 4 else (2 if n >= 2 else 1)

cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
    enable_lora=True, max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp, startup_timeout=60 * 20)
server = VllmServer(cfg, adapter_manifest=adapters)
server.start()
models = server.create_model_registry(aliases=[declared, TARGET], model_prefix='openai/', api_key='EMPTY')

## 6. Claim vs. proof loop

For every task: the agent's *claim* is "I produced a patch". The *proof* is the evaluator's physical verdict (`resolved`). A **divergence** is claim without proof. This is the measurable false-positive signal, computed from real runs only.

`GATE = True` applies the C-T rule at submission time: patches without proof are replaced by an empty prediction.

In [ ]:
import asyncio, concurrent.futures, time, uuid
import pandas as pd, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

N_TASKS = 5      # raise for a real A/B run
GATE = False     # True = submit only proven patches

def run_sync(fn, *a, **kw):
    def go(): return asyncio.run(fn(*a, **kw))
    try: asyncio.get_running_loop()
    except RuntimeError: return go()
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(go).result()

ev = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
ev = ev.get('evaluation', ev)
limits, gen = build_submission_limits()
turns = ev.get('max_turns', ev.get('max_llm_calls'))

evaluator = Evaluator(EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
    timeout_seconds=int(ev.get('timeout_seconds', 300)), max_time_minutes=float(ev.get('max_time_minutes', 60.0)),
    max_tool_calls=int(ev.get('max_tool_calls', 100)), max_turns=int(turns) if turns is not None else None,
    limits=limits, generation_constraints=gen, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                                                    token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / 'wheels', verbose=False))

rows, preds = [], []
for i, task in enumerate(tasks[:N_TASKS], 1):
    trace = uuid.uuid4().hex
    span = uuid.uuid4().hex[:16]
    T.emit('STEP_STARTED', CORE, trace_id=trace, span_id=span, data={'task': task.instance_id})
    t0 = time.time()
    try:
        r = run_sync(evaluator.evaluate_task, task=task, task_index=i, total_tasks=N_TASKS)
        patch = r.agent_patch or ''
        proof = {'resolved': bool(r.resolved), 'exit_code': r.test_exit_code, 'tool_calls': r.tool_calls}
        T.emit('STEP_COMPLETED', CORE, trace_id=trace, span_id=span,
               data={'task': task.instance_id, 'claim_patch_chars': len(patch), 'proof': proof},
               metrics={'seconds': round(time.time() - t0, 1)})
    except Exception as exc:
        patch, proof = '', {'resolved': False, 'exit_code': -1, 'tool_calls': 0}
        T.emit('STEP_FAILED', CORE, trace_id=trace, span_id=span,
               data={'task': task.instance_id, 'error': repr(exc)[:300]})
    claim = bool(patch.strip())
    diverged = claim and not proof['resolved']
    rows.append({'id': task.instance_id, 'claim': claim, 'resolved': proof['resolved'],
                 'diverged': diverged, 'exit_code': proof['exit_code'], 'tool_calls': proof['tool_calls']})
    preds.append({'id': task.instance_id, 'prediction': patch if (not GATE or proof['resolved']) else ''})
    print(rows[-1])

report = pd.DataFrame(rows)
submission_df = pd.DataFrame(preds, columns=['id', 'prediction'])
claims = int(report.claim.sum())
print(f"\nresolved {int(report.resolved.sum())}/{len(report)}; "
      f"claims {claims}; claim-without-proof {int(report.diverged.sum())} "
      f"({(report.diverged.sum() / claims * 100 if claims else 0):.1f}% of claims)")
display(report)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.AGENT_CONFIG is enabled.
  check_feature_enabled()
/tmp/ipykernel_163/2207752231.py:30: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [  19.8s] 💭 (986 chars) "The user wants to "Automate release preparation".
Looking at the problem statement, it seems like a PR description rathe..."
  [  19.8s] 🔧 run_command find . -maxdepth 2 -name "*release*"
  [  19.8s]   → ./scripts/add_latest_release_date.py
  [  21.5s] 💭 (86 chars) "Okay, there is `./scripts/add_latest_release_date.py`.
Let's see what's in `scripts/`."
  [  21.5s] 🔧 run_command ls scripts/
  [  21.5s]   → add_latest_release_date.py
  [  25.6s] 💭 (338 chars) "The problem statement is very vague. "Automate release preparation".
If I look at the `asyncer` PR 608 (if I could), I m..."
  [  25.6s] 📖 read_file (scripts/add_latest_release_date.py)
  [  25.6s]   ✓ {'status': 'ok', 'filepath': 'scripts/add_latest_release_date.py', 'content': '"""Check 
release-notes.md and add today\'…
  [  31.3s] 💭 (530 chars) "This script just adds the date to the release notes.

The problem statement says "Automate release preparation".
Maybe i..."
  [  31.3s] 🔧 run_command grep -r "release" . | grep ".py" | head -n 20
  [  31.3s]   → ./scripts/add_latest_release_date.py:"""Check release-notes.md and add today's date to the latest 
release header if miss
  [  36.9s] 💭 (479 chars) "I see `.pre-commit-config.yaml` has `uv run python scripts/add_latest_release_date.py`.

If the task is to "Automate rel..."
  [  36.9s] 📖 read_file (pyproject.toml)
  [  36.9s]   ✓ {'status': 'ok', 'filepath': 'pyproject.toml', 'content': '[build-system]\nrequires = 
["pdm-backend"]\nbuild-backend = "…
  [  40.4s] 💭 (147 chars) "`dynamic = ["version"]` in `pyproject.toml`. This means the version is likely defined 
in `fastapi/__init__.py`.

Let's c..."
  [  40.4s] 📖 read_file (fastapi/__init__.py)
  [  40.4s]   ✓ {'status': 'ok', 'filepath': 'fastapi/__init__.py', 'content': '"""FastAPI framework, high 
performance, easy to learn, f…
  [  49.3s] 💭 (861 chars) "Yes, `__version__ = "0.136.3"`.

Now, what does "Automate release preparation" mean?
If I look at the `asyncer` PR 608, ..."
  [  49.3s] 🔧 run_command grep -r "asyncer" .
  [  49.6s]   → ./fastapi/.agents/skills/fastapi/references/other-tools.md:uv add asyncer

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   🔧 run_command                                                                                                │
│     ... (11 earlier lines)                                                                                      │
│     ./docs/zh-hant/docs/async.md:我另外在 AnyIO                                                                 │
│ 之上做了一個薄封裝的函式庫，稍微改進型別註解以獲得更好的**自動補全**、**即時錯誤**等。同時它也提供友善的介紹與  │
│ 教學，幫助你**理解**並撰寫**自己的非同步程式碼**：[Asyncer](https://asyncer.tiangolo.com/)。當你需要**將非同步  │
│ 程式碼與一般**（阻塞/同步）**程式碼整合**時，它特別實用。                                                       │
│     ./docs/fr/docs/advanced/stream-data.md:Si vous devez appeler du code bloquant depuis une fonction async, ou │
│ une fonction async depuis une fonction bloquante, vous pouvez utiliser [Asyncer](https://asyncer.tiangolo.com), │
│ une bibliothèque sœur de FastAPI.                                                                               │
│     ./docs/fr/docs/async.md:J’ai créé une autre bibliothèque au-dessus d’AnyIO, comme une fine surcouche, pour  │
│ améliorer un peu les annotations de type et obtenir une meilleure **autocomplétion**, des **erreurs en ligne**, │
│ etc. Elle propose également une introduction et un tutoriel accessibles pour vous aider à **comprendre** et     │
│ écrire **votre propre code async** : [Asyncer](https://asyncer.tiangolo.com/). Elle sera particulièrement utile │
│ si vous devez **combiner du code async avec du code classique** (bloquant/synchrone).          

{'id': 'fastapi_15661', 'claim': False, 'resolved': False, 'diverged': False, 'exit_code': -1, 'tool_calls': 7}


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   5.2s] 💭 (301 chars) "The user wants to validate Server Sent Event (SSE) fields in FastAPI to prevent sending
broken data (specifically multi-..."
  [   5.2s] 🤖 search_similar_code {"query": "Server Sent Events"}
  [   6.4s]   → search_similar_code: {"status": "ok", "query": "Server Sent Events", "results": [], "count": 0, 
"is_truncated": false}
  [  11.0s] 💭 (497 chars) "Maybe it's not explicitly called "Server Sent Events" in the code. I'll try searching 
for "EventSource" or "SSE".
Actual..."
  [  11.0s] 🔧 run_command grep -rn "SSE" .
  [  11.2s]   → ./fastapi/.agents/skills/fastapi/references/streaming.md:14:## Server-Sent Events (SSE)
  [  17.0s] 💭 (410 chars) "The grep results point to `fastapi/sse.py` as a key file.
Specifically:
`./fastapi/sse.py:38:        raise ValueError(\"..."
  [  17.0s] 📖 read_file (fastapi/sse.py)
  [  17.0s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, 
Any\n\nfrom annotated_doc impor…

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   📖 read_file                                                                                                  │
│                                                                                                                 │
│   Path:                                                                                                         │
│     fastapi/sse.py                                                                                              │
│                                                                                                                 │
│   Response:                                                                                                     │
│     {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, Any\n\nfrom        │
│ annotated_doc import Doc\nfrom pydantic import AfterValidator, BaseModel, Field, model_validator\nfrom          │
│ starlette.responses import StreamingResponse\n\n# Canonical SSE event schema matching the OpenAPI 3.2 spec\n#   │
│ (Section 4.14.4 "Special Considerations for Server-Sent Events")\n_SSE_EVENT_SCHEMA: dict[str, Any] = {\n       │
│ "type": "object",\n    "properties": {\n        "data": {"type": "string"},\n        "event": {"type":          │
│ "string"},\n        "id": {"type": "string"},\n        "retry": {"type": "integer", "minimum": 0},\n            │
│ },\n}\n\n\nclass EventSourceResponse(StreamingResponse):\n    """Streaming response with `text/event-stream`    │
│ media type.\n\n    Use as `response_class=EventSourceResponse` on a *path operation* that uses `yield`\n    to  │
│ enable Server Sent Events (SSE) responses.\n\n    Works with **any HTTP method** (`GET`, `POST`, etc.), which   │
│ makes it compatible\n    with protocols like MCP that stream SSE over `POST`.\n\n    The actual encoding logic  │
│ lives in the FastAPI routing layer. This class\n    serves mainly as a marker and sets the correct              │
│ `Content-Type`.\n    """\n\n    media_type = "text/event-stream"\n\n\ndef _check_id_no_null(v: str | None) ->   │
│ str | None:\n    if v is not None and "\\0" in v:\n        raise ValueError("SSE \'id\' must not contain null   │
│ characters")\n    return v\n\n\nclass ServerSentEvent(BaseModel):\n    """Represents a single Server-Sent       │
│ Event.\n\n    When `yield`ed from a *path operation function* that uses\n                                       │
│ `response_class=EventSourceResponse`, each `ServerSentEvent` is encoded\n    into the [SSE wire                 │
│ format](https://html.spec.whatwg.org/multipage/server-sent-events.html#parsing-an-event-st

{'id': 'fastapi_15588', 'claim': False, 'resolved': False, 'diverged': False, 'exit_code': -1, 'tool_calls': 3}


...

[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   6.7s] 💭 (622 chars) "The user wants to add support for Server Sent Events (SSE) in FastAPI.
I need to find where to implement this. SSE typic..."
  [   6.7s] 🤖 search_similar_code {"query": "StreamingResponse"}
  [   7.9s]   → search_similar_code: {"status": "ok", "query": "StreamingResponse", "results": [], "count": 0, 
"is_truncated": false}
  [  13.0s] 💭 (560 chars) "That's unexpected. `StreamingResponse` is a core part of FastAPI (inherited from 
Starlette). Let me search for where res..."
  [  13.0s] 🔧 run_command find . -maxdepth 2 -not -path '*/.*'
  [  13.0s]   → .
  [  15.8s] 💭 (61 chars) "Okay, I see `fastapi/responses.py`. Let me check its content."
  [  15.8s] 📖 read_file (fastapi/responses.py)
  [  15.8s]   ✓ {'status': 'ok', 'filepath': 'fastapi/responses.py', 'content': 'from typing import Any\n\nfrom 
fastapi.exceptions impor…
  [  41.7s] 💭 (2871 chars) "FastAPI's `responses.py` mostly just re-exports Starlette's responses.
To add SSE support, I should probably add a `SSER..."
  [  41.7s] 🤖 search_similar_code {"query": "text/event-stream"}
  [  41.7s]   → search_similar_code: {"status": "ok", "query": "text/event-stream", "results": [], "count": 0, 
"is_truncated": false}

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   🤖 search_similar_code                                                                                        │
│                                                                                                                 │
│   Args:                                                                                                         │
│     {                                                                                                           │
│       "query": "text/event-stream"                                                                              │
│     }                                                                                                           │
│                                                                                                                 │
│   Response:                                                                                                     │
│     {                                                                                                           │
│       "status": "ok",                                                                                           │
│       "query": "text/event-stream",                                                                             │
│       "results": [],                                                                                            │
│       "count": 0,                                                                                               │
│       "is_truncated": false                                                                                     │
│     }                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────── SWE-gemma Agent Evaluator ───────────────────────────────────────────╮
│   [4/5] fastapi_15030 (fastapi/fastapi)  ·  1m 00s                                                              │
│                                                                                                                 │
│           Tools ━━━━━━━━╸━━━━━━━━━━━━━   4 / 10  (6 left)                                                       │
│            Cost ━━━━━━━━━━━━━━━━━━━━━━   $0.00 (Unlimited)                                    

{'id': 'fastapi_15030', 'claim': False, 'resolved': False, 'diverged': False, 'exit_code': -1, 'tool_calls': 4}


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   8.0s] 💭 (808 chars) "The user wants to implement JSON response serialization with Pydantic in Rust when 
there's a Pydantic return type or res..."
  [   8.0s] 🤖 search_similar_code {"query": "serialize response pydantic"}
  [   9.1s]   → search_similar_code: {"status": "ok", "query": "serialize response pydantic", "results": [], 
"count": 0, "is_truncated": false}
  [  12.0s] 💭 (234 chars) "Okay, that didn't return anything. Let's try searching for "response_model" or 
"serialize".
FastAPI's response serializa..."
  [  12.0s] 🔧 run_command find fastapi -name "*routing*"
  [  12.0s]   → fastapi/routing.py
  [  13.2s] 💭 (35 chars) "Let's examine `fastapi/routing.py`."
  [  13.2s] 📖 read_file (fastapi/routing.py)
  [  13.3s]   ✓ {'status': 'ok', 'filepath': 'fastapi/routing.py', 'content': 'import contextlib\nimport 
email.message\nimport functools…
  [  16.0s] 💭 (71 chars) "The `fastapi/routing.py` file is very large. I need to find `APIRoute`."
  [  16.0s] 🔧 run_command grep -n "class APIRoute" fastapi/routing.py
  [  16.0s]   → 554:class APIRoute(routing.Route):
  [  18.1s] 💭 (73 chars) "Let's look at `APIRoute` and its methods, especially `get_route_handler`."
  [  18.1s] 📖 read_file (fastapi/routing.py)
  [  18.2s]   ✓ {'status': 'ok', 'filepath': 'fastapi/routing.py', 'content': 'class APIRoute(routing.Route):\n    
def __init__(\n      …
  [  23.9s] 💭 (292 chars) "`APIRoute.get_route_handler` calls `get_request_handler`. Let's find 
`get_request_handler`.
It's likely in `fastapi/rout..."
  [  23.9s] 🔧 run_command grep -n "def get_request_handler" fastapi/routing.py
  [  23.9s]   → 317:def get_request_handler(
  [  26.0s] 💭 (57 chars) "Let's read `get_request_handler` in `fastapi/routing.py`."
  [  26.0s] 📖 read_file (fastapi/routing.py)
  [  26.0s]   ✓ {'status': 'ok', 'filepath': 'fastapi/routing.py', 'content': 'def get_request_handler(\n    
dependant: Dependant,\n    …
  [  29.7s] 💭 (135 chars) "I see `serialize_response` being called. Let's find where `serialize_response` is 
defined.
It's likely in `fastapi/routi..."
  [  29.7s] 🔧 run_command grep -n "def serialize_response" fastapi/routing.py
  [  29.7s]   → 262:async def serialize_response(
  [  31.5s] 💭 (32 chars) "Let's read `serialize_response`."
  [  31.5s] 📖 read_file (fastapi/routing.py)
  [  31.5s]   ✓ {'status': 'ok', 'filepath': 'fastapi/routing.py', 'content': 'async def serialize_response(\n    
*,\n    field: ModelFi…
  [  39.5s] 💭 (837 chars) "The `serialize_response` function uses `field.validate` and `field.serialize`.
The problem statement says "Serialize JSO..."
  [  39.5s] 📖 read_file (fastapi/_compat.py)
  [  39.5s]   ❌ {'status': 'error', 'error_type': 'FileReadError', 'error_message': 'Source path 
/workspace/fastapi/_compat.py not found…
  [  40.9s] 💭 (47 chars) "Ah, let me check the directory structure again."
  [  40.9s] 🔧 run_command find fastapi -name "_compat.py"
  [  40.9s]   → error (exit 0): 

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   🔧 run_command                                                                                                │
│                                                                                                                 │
│   Command:                                                                                                      │
│     find fastapi -name "_compat.py"                                                                             │
│                                                                                                                 │
│   Error:                                                                                                 

{'id': 'fastapi_14962', 'claim': False, 'resolved': False, 'diverged': False, 'exit_code': -1, 'tool_calls': 10}

resolved 0/5; claims 0; claim-without-proof 0 (0.0% of claims)


,id,claim,resolved,diverged,exit_code,tool_calls
0,fastapi_15661,False,False,False,-1,7
1,fastapi_15588,False,False,False,-1,3
2,fastapi_15589,False,False,False,-1,10
3,fastapi_15030,False,False,False,-1,4
4,fastapi_14962,False,False,False,-1,10


## 7. Audit: seal the log, verify, observe

In [ ]:
T.emit('CORE_STOPPED', CORE)
events = A.read_events(LOG)
CHAIN = LOG.with_suffix('.chain.jsonl')
head = A.build_chain(events, CHAIN)
ok, msg = A.verify_chain(events, CHAIN)
level, issues = A.observe(events)
print(f'{len(events)} events | chain: {ok} ({msg}) | observer exit={level} {issues}')

# tamper test: change one event in memory, the chain must reject it
bad = [dict(e) for e in events]; bad[1]['core_id'] = 'forged'
print('tamper check ->', A.verify_chain(bad, CHAIN))

12 events | chain: True (ok, head=c89e511c0648) | observer exit=0 []
tamper check -> (False, 'tamper detected at event #1')


## 8. Package submission

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

zip_path = Path(shutil.make_archive(str(WORKING_DIR / 'submission'), 'zip', root_dir=AGENT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    infos = zf.infolist()
    assert sum(i.file_size for i in infos) <= MAX_SUBMISSION_SIZE_BYTES, 'archive too large'
    for i in infos:
        if not i.is_dir():
            assert Path(i.filename).suffix.lower() in ALLOWED_SUBMISSION_EXTENSIONS, f'bad extension: {i.filename}'
print(f'{zip_path.name}: {zip_path.stat().st_size / 2**20:.2f} MiB')

submission.zip: 0.17 MiB


In [4]:
import sys
import os
from pathlib import Path

print("=== System Paths and Files ===")
print("Current working directory:", os.getcwd())
print("Contents of root /kaggle if exists:")
kaggle_path = Path('/kaggle')
if kaggle_path.exists():
    for p in kaggle_path.rglob('*'):
        if p.is_file() and ('eval_config' in p.name or 'agent.yaml' in p.name):
            print("Found file in kaggle:", p)
else:
    print("/kaggle directory does not exist.")

print("\nContents of /content:")
for p in Path('/content').rglob('*'):
    if p.is_file() and ('eval_config' in p.name or 'agent.yaml' in p.name):
        print("Found file in content:", p)

# Also let's check sys.path to see where libraries are installed
print("\nsys.path:", sys.path)

=== System Paths and Files ===
Current working directory: /content
Contents of root /kaggle if exists:

Contents of /content:

sys.path: ['/kaggle/working/chronos', '/kaggle/working', '/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython']


### Корректировка системного промпта агента

Ниже представлен исправленный системный промпт для `sample_submission/prompts/system.md`, который устраняет проблемы с неэффективным поиском и превышением лимитов.

In [5]:
new_prompt = """You are an elite software engineering agent. Your task is to resolve issues in Python repositories by locating bugs, writing precise patches, and submitting them.

### CRITICAL BUDGET AND SEARCH RULES:

1. **Search and Symbol Retrieval Rules:**
   - When using the `search_similar_code` tool, DO NOT pass natural language phrases, sentences, or high-level descriptions (e.g., "Server Sent Events" or "asyncer implementation").
   - This tool ONLY works with concrete symbol names. Pass only specific class names, function names, or variable names (e.g., `SseEventParser` or `get_event_stream`).

2. **Grep and File Filtering Rules:**
   - NEVER perform recursive unconstrained searches across the entire repository (like `grep -r "pattern" .`). This wastes tool calls on docs, tests, and translation files.
   - Always constraint your search using exact patterns, target specific source directories (e.g., `fastapi/`), and strictly filter files (e.g., `grep -rn --include=*.py "pattern" fastapi/`).

3. **Hard Stop and Patch Submission Budget:**
   - You have a strict limit of 10 tool calls, but you MUST aim to resolve the issue much earlier to avoid timeouts.
   - **Strict Rule:** If you reach your **6th tool call / turn** and have not yet submitted a patch, you MUST stop all further exploration immediately.
   - Prepare the best possible patch with what you currently know, apply it, and make your very last call `submit_patch`. A partial or weak patch is infinitely better than no patch at all (which results in a timeout/zero score).

4. **Execution Flow:**
   - Locate the bug using optimized class/function searches.
   - Verify using restricted grep on source files if needed.
   - Apply changes.
   - Call `submit_patch` before your budget runs out.
"""

# Записываем промпт в директорию агента
prompt_path = AGENT_DIR / 'prompts' / 'system.md'
try:
    prompt_path.parent.mkdir(parents=True, exist_ok=True)
    prompt_path.write_text(new_prompt, encoding='utf-8')
    print(f"Successfully wrote updated system prompt to: {prompt_path}")
except Exception as e:
    print(f"Could not write to {prompt_path}: {e}. (Make sure to run setup cells first to initialize AGENT_DIR)")


Successfully wrote updated system prompt to: /content/sample_submission/prompts/system.md


In [6]:
from pathlib import Path

prompt_file = Path('/content/sample_submission/prompts/system.md')
if prompt_file.exists():
    print("--- File content of system.md ---")
    print(prompt_file.read_text(encoding='utf-8'))
else:
    print("File does not exist at /content/sample_submission/prompts/system.md")

--- File content of system.md ---
You are an elite software engineering agent. Your task is to resolve issues in Python repositories by locating bugs, writing precise patches, and submitting them.

### CRITICAL BUDGET AND SEARCH RULES:

1. **Search and Symbol Retrieval Rules:**
   - When using the `search_similar_code` tool, DO NOT pass natural language phrases, sentences, or high-level descriptions (e.g., "Server Sent Events" or "asyncer implementation"). 
   - This tool ONLY works with concrete symbol names. Pass only specific class names, function names, or variable names (e.g., `SseEventParser` or `get_event_stream`).

2. **Grep and File Filtering Rules:**
   - NEVER perform recursive unconstrained searches across the entire repository (like `grep -r "pattern" .`). This wastes tool calls on docs, tests, and translation files.
   - Always constraint your search using exact patterns, target specific source directories (e.g., `fastapi/`), and strictly filter files (e.g., `grep -rn --i

In [7]:
from pathlib import Path

agent_yaml = Path('/content/sample_submission/agent.yaml')
if agent_yaml.exists():
    print("--- File content of agent.yaml ---")
    print(agent_yaml.read_text(encoding='utf-8'))
else:
    print("File does not exist at /content/sample_submission/agent.yaml")

File does not exist at /content/sample_submission/agent.yaml


In [8]:
from pathlib import Path

agent_yaml = Path('/content/sample_submission/agent.yaml')
if agent_yaml.exists():
    print("--- File content of agent.yaml ---")
    print(agent_yaml.read_text(encoding='utf-8'))
else:
    print("File does not exist at /content/sample_submission/agent.yaml")

File does not exist at /content/sample_submission/agent.yaml


In [9]:
from pathlib import Path

agent_dir = Path('/content/sample_submission')
if agent_dir.exists():
    yaml_files = list(agent_dir.rglob('*.yaml')) + list(agent_dir.rglob('*.yml'))
    print(f'Found {len(yaml_files)} YAML files:')
    for f in yaml_files:
        print(f'- {f.relative_to(agent_dir)} ({f})')
else:
    print('/content/sample_submission directory does not exist.')

Found 0 YAML files:


In [10]:
from pathlib import Path

agent_dir = Path('/content/sample_submission')
if agent_dir.exists():
    yaml_files = list(agent_dir.rglob('*.yaml')) + list(agent_dir.rglob('*.yml'))
    print(f"Found {len(yaml_files)} YAML files:")
    for f in yaml_files:
        print(f"- {f.relative_to(agent_dir)} ({f})")
else:
    print("/content/sample_submission directory does not exist.")

Found 0 YAML files:


In [11]:
from pathlib import Path
import shutil

source_dir = Path('/kaggle/input/competitions/gemma-4-developer-agent/sample_submission')
target_dir = Path('/content/sample_submission')

if source_dir.exists():
    print('Original sample_submission found. Restoring configuration files...')
    for p in source_dir.rglob('*'):
        if p.is_file():
            rel_path = p.relative_to(source_dir)
            dest = target_dir / rel_path
            if rel_path == Path('prompts/system.md'):
                # Пропускаем, чтобы сохранить наши изменения в prompts/system.md
                continue
            if not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dest)
                print(f'Restored: {rel_path}')
else:
    print('Original source directory does not exist. We may need to verify the environment paths.')

Original source directory does not exist. We may need to verify the environment paths.


In [12]:
from pathlib import Path
import shutil

# Проверяем оригинальный путь к данным
source_dir = Path('/kaggle/input/competitions/gemma-4-developer-agent/sample_submission')
target_dir = Path('/content/sample_submission')

if source_dir.exists():
    print("Original sample_submission found at /kaggle. List of files:")
    for p in source_dir.rglob('*'):
        if p.is_file():
            print(f"- {p.relative_to(source_dir)} ({p.stat().st_size} bytes)")

    # При необходимости восстановим отсутствующие файлы (кроме prompts/system.md, который мы уже обновили)
    for p in source_dir.rglob('*'):
        if p.is_file():
            rel_path = p.relative_to(source_dir)
            dest = target_dir / rel_path
            if rel_path == Path('prompts/system.md'):
                # Не перезаписываем наш новый системный промпт
                continue
            if not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dest)
                print(f"Restored: {rel_path}")
else:
    print("Original directory at /kaggle does not exist. Let's check alternative locations.")

Original directory at /kaggle does not exist. Let's check alternative locations.


In [13]:
import os
from pathlib import Path

print("Searching for 'agent.yaml' or any config templates across the system...")
found = []
for root, dirs, files in os.walk('/'):
    # Исключаем системные и виртуальные папки для ускорения поиска
    if any(p in root for p in ['/proc', '/sys', '/dev', '/var/lib/docker', '/usr/local/lib']):
        continue
    for file in files:
        if file == 'agent.yaml' or 'sampling.yaml' in file:
            full_path = Path(root) / file
            found.append(full_path)
            print(f"Found: {full_path}")

if not found:
    print("No configuration files found. Let's see what is inside /content/sample_submission and if we can reconstruct agent.yaml manually.")

Searching for 'agent.yaml' or any config templates across the system...
No configuration files found. Let's see what is inside /content/sample_submission and if we can reconstruct agent.yaml manually.


In [14]:
import os
from pathlib import Path

print("Searching for 'agent.yaml' or any config templates across the system...")
found = []
for root, dirs, files in os.walk('/'):
    # Исключаем системные и виртуальные папки для ускорения поиска
    if any(p in root for p in ['/proc', '/sys', '/dev', '/var/lib/docker', '/usr/local/lib']):
        continue
    for file in files:
        if file == 'agent.yaml' or 'sampling.yaml' in file:
            full_path = Path(root) / file
            found.append(full_path)
            print(f"Found: {full_path}")

if not found:
    print("No configuration files found. Let's see what is inside /content/sample_submission and if we can reconstruct agent.yaml manually.")

Searching for 'agent.yaml' or any config templates across the system...
No configuration files found. Let's see what is inside /content/sample_submission and if we can reconstruct agent.yaml manually.


In [15]:
import yaml
from pathlib import Path

agent_dir = Path('/content/sample_submission')
agent_dir.mkdir(parents=True, exist_ok=True)

# 1. Reconstruct agent.yaml
agent_yaml_content = """
agent_class: GemmaDeveloperAgent
model: gemma-4-31b-it-qat-w4a16-ct
max_turns: 8
prompt_path: prompts/system.md
sub_agents:
  - sub_agents/code_analyzer.yaml
"""
(agent_dir / 'agent.yaml').write_text(agent_yaml_content.strip(), encoding='utf-8')

# 2. Reconstruct sub_agents/code_analyzer.yaml
sub_agent_dir = agent_dir / 'sub_agents'
sub_agent_dir.mkdir(exist_ok=True)
sub_agent_content = """
agent_class: CodeAnalyzerSubAgent
model: gemma-4-31b-it-qat-w4a16-ct
prompt_path: prompts/analyzer.md
"""
(sub_agent_dir / 'code_analyzer.yaml').write_text(sub_agent_content.strip(), encoding='utf-8')

# 3. Reconstruct prompts/analyzer.md
prompts_dir = agent_dir / 'prompts'
prompts_dir.mkdir(exist_ok=True)
analyzer_prompt_content = """Your task is to analyze the repository code and identify the locations of bugs. Help the main agent make precise changes."""
if not (prompts_dir / 'analyzer.md').exists():
    (prompts_dir / 'analyzer.md').write_text(analyzer_prompt_content.strip(), encoding='utf-8')

# 4. Reconstruct configs/sampling.yaml with optimized thinking_budget
configs_dir = agent_dir / 'configs'
configs_dir.mkdir(exist_ok=True)
sampling_content = """temperature: 0.2
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 1024
  include_thoughts: true
"""
(configs_dir / 'sampling.yaml').write_text(sampling_content.strip(), encoding='utf-8')

# 5. Reconstruct eval_config.yaml
eval_config_content = """evaluation:
  max_turns: 8
  timeout_seconds: 300
  max_time_minutes: 60.0
  max_tool_calls: 100
"""
(agent_dir / 'eval_config.yaml').write_text(eval_config_content.strip(), encoding='utf-8')

print('Successfully reconstructed all configuration files in /content/sample_submission!')
for p in sorted(agent_dir.rglob('*')):
    if p.is_file():
        print(f'- {p.relative_to(agent_dir)} ({p.stat().st_size} bytes)')

Successfully reconstructed all configuration files in /content/sample_submission!
- agent.yaml (157 bytes)
- configs/sampling.yaml (119 bytes)
- eval_config.yaml (96 bytes)
- prompts/analyzer.md (121 bytes)
- prompts/system.md (1765 bytes)
- sub_agents/code_analyzer.yaml (101 bytes)


In [16]:
import yaml
from pathlib import Path

agent_dir = Path('/content/sample_submission')
agent_dir.mkdir(parents=True, exist_ok=True)

# 1. Воссоздаем agent.yaml
agent_yaml_content = """
agent_class: GemmaDeveloperAgent
model: gemma-4-31b-it-qat-w4a16-ct
max_turns: 8
prompt_path: prompts/system.md
sub_agents:
  - sub_agents/code_analyzer.yaml
"""
(agent_dir / 'agent.yaml').write_text(agent_yaml_content.strip(), encoding='utf-8')

# 2. Воссоздаем sub_agents/code_analyzer.yaml
sub_agent_dir = agent_dir / 'sub_agents'
sub_agent_dir.mkdir(exist_ok=True)
sub_agent_content = """
agent_class: CodeAnalyzerSubAgent
model: gemma-4-31b-it-qat-w4a16-ct
prompt_path: prompts/analyzer.md
"""
(sub_agent_dir / 'code_analyzer.yaml').write_text(sub_agent_content.strip(), encoding='utf-8')

# 3. Воссоздаем prompts/analyzer.md
prompts_dir = agent_dir / 'prompts'
prompts_dir.mkdir(exist_ok=True)
analyzer_prompt_content = """Your task is to analyze the repository code and identify the locations of bugs. Help the main agent make precise changes."""
if not (prompts_dir / 'analyzer.md').exists():
    (prompts_dir / 'analyzer.md').write_text(analyzer_prompt_content.strip(), encoding='utf-8')

# 4. Воссоздаем configs/sampling.yaml с оптимизированным бюджетом размышлений
configs_dir = agent_dir / 'configs'
configs_dir.mkdir(exist_ok=True)
sampling_content = """temperature: 0.2
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 1024
  include_thoughts: true
"""
(configs_dir / 'sampling.yaml').write_text(sampling_content.strip(), encoding='utf-8')

# 5. Воссоздаем eval_config.yaml
eval_config_content = """evaluation:
  max_turns: 8
  timeout_seconds: 300
  max_time_minutes: 60.0
  max_tool_calls: 100
"""
(agent_dir / 'eval_config.yaml').write_text(eval_config_content.strip(), encoding='utf-8')

print("Successfully reconstructed all configuration files in /content/sample_submission!")
for p in sorted(agent_dir.rglob('*')):
    if p.is_file():
        print(f"- {p.relative_to(agent_dir)} ({p.stat().st_size} bytes)")

Successfully reconstructed all configuration files in /content/sample_submission!
- agent.yaml (157 bytes)
- configs/sampling.yaml (119 bytes)
- eval_config.yaml (96 bytes)
- prompts/analyzer.md (121 bytes)
- prompts/system.md (1765 bytes)
- sub_agents/code_analyzer.yaml (101 bytes)


In [ ]:
import yaml
from pathlib import Path

sampling_path = Path('/content/sample_submission/configs/sampling.yaml')

if sampling_path.exists():
    try:
        config_data = yaml.safe_load(sampling_path.read_text(encoding='utf-8'))
        print("=== Валидация sampling.yaml ===\n")
        print("Конфигурация успешно загружена:\n")
        print(yaml.dump(config_data, default_flow_style=False, allow_unicode=True))

        # Проверки критических параметров
        thinking_config = config_data.get('thinking_config', {})
        budget = thinking_config.get('thinking_budget')

        if budget is not None:
            print(f"[OK] Параметр thinking_budget найден: {budget}")
            if budget > 2048:
                print("[WARNING] Бюджет размышлений довольно высокий, что может вызвать таймауты при работе vLLM в ограниченном окружении.")
            else:
                print("[INFO] Бюджет размышлений оптимизирован для предотвращения таймаутов.")
        else:
            print("[WARNING] Параметр thinking_budget отсутствует или задан неверно.")

    except Exception as e:
        print(f"[ERROR] Ошибка синтаксиса YAML или чтения файла: {e}")
else:
    print(f"[ERROR] Файл конфигурации не найден по пути: {sampling_path}")

In [ ]:
from pathlib import Path

agent_yaml_path = Path('/content/sample_submission/agent.yaml')

if agent_yaml_path.exists():
    print("=== Содержимое agent.yaml ===\n")
    print(agent_yaml_path.read_text(encoding='utf-8'))
else:
    print(f"[ERROR] Файл не найден по пути: {agent_yaml_path}")

In [1]:
from pathlib import Path
import yaml

path = Path('/content/sample_submission/agent.yaml')
if path.exists():
    content = path.read_text(encoding='utf-8')
    print("=== agent.yaml ===")
    print(content)
    try:
        parsed = yaml.safe_load(content)
        print("\n[OK] Файл валиден как YAML")
    except Exception as e:
        print(f"\n[ERROR] Ошибка синтаксиса YAML: {e}")
else:
    print("[ERROR] Файл agent.yaml не найден")

[ERROR] Файл agent.yaml не найден


In [2]:
import os
from pathlib import Path

# Принудительно воссоздаем файлы, так как окружение было очищено или ячейки генерации не запускались
agent_dir = Path('/content/sample_submission')
agent_dir.mkdir(parents=True, exist_ok=True)

# 1. Reconstruct agent.yaml
agent_yaml_content = """
agent_class: GemmaDeveloperAgent
model: gemma-4-31b-it-qat-w4a16-ct
max_turns: 8
prompt_path: prompts/system.md
sub_agents:
  - sub_agents/code_analyzer.yaml
"""
(agent_dir / 'agent.yaml').write_text(agent_yaml_content.strip(), encoding='utf-8')

# 2. Reconstruct sub_agents/code_analyzer.yaml
sub_agent_dir = agent_dir / 'sub_agents'
sub_agent_dir.mkdir(exist_ok=True)
sub_agent_content = """
agent_class: CodeAnalyzerSubAgent
model: gemma-4-31b-it-qat-w4a16-ct
prompt_path: prompts/analyzer.md
"""
(sub_agent_dir / 'code_analyzer.yaml').write_text(sub_agent_content.strip(), encoding='utf-8')

# 3. Reconstruct prompts/analyzer.md
prompts_dir = agent_dir / 'prompts'
prompts_dir.mkdir(exist_ok=True)
analyzer_prompt_content = """Your task is to analyze the repository code and identify the locations of bugs. Help the main agent make precise changes."""
(prompts_dir / 'analyzer.md').write_text(analyzer_prompt_content.strip(), encoding='utf-8')

# 4. Reconstruct prompts/system.md
system_prompt_content = """You are an elite software engineering agent. Your task is to resolve issues in Python repositories by locating bugs, writing precise patches, and submitting them.

### CRITICAL BUDGET AND SEARCH RULES:

1. **Search and Symbol Retrieval Rules:**
   - When using the `search_similar_code` tool, DO NOT pass natural language phrases, sentences, or high-level descriptions (e.g., "Server Sent Events" or "asyncer implementation").
   - This tool ONLY works with concrete symbol names. Pass only specific class names, function names, or variable names (e.g., `SseEventParser` or `get_event_stream`).

2. **Grep and File Filtering Rules:**
   - NEVER perform recursive unconstrained searches across the entire repository (like `grep -r "pattern" .`). This wastes tool calls on docs, tests, and translation files.
   - Always constraint your search using exact patterns, target specific source directories (e.g., `fastapi/`), and strictly filter files (e.g., `grep -rn --include=*.py "pattern" fastapi/`).

3. **Hard Stop and Patch Submission Budget:**
   - You have a strict limit of 10 tool calls, but you MUST aim to resolve the issue much earlier to avoid timeouts.
   - **Strict Rule:** If you reach your **6th tool call / turn** and have not yet submitted a patch, you MUST stop all further exploration immediately.
   - Prepare the best possible patch with what you currently know, apply it, and make your very last call `submit_patch`.
"""
(prompts_dir / 'system.md').write_text(system_prompt_content.strip(), encoding='utf-8')

# 5. Reconstruct configs/sampling.yaml
configs_dir = agent_dir / 'configs'
configs_dir.mkdir(exist_ok=True)
sampling_content = """temperature: 0.2
top_p: 0.95
max_output_tokens: 16384
thinking_config:
  thinking_budget: 1024
  include_thoughts: true
"""
(configs_dir / 'sampling.yaml').write_text(sampling_content.strip(), encoding='utf-8')

print("[SUCCESS] Все конфигурационные файлы успешно воссозданы в /content/sample_submission!")

[SUCCESS] Все конфигурационные файлы успешно воссозданы в /content/sample_submission!


In [3]:
from pathlib import Path
import yaml

path = Path('/content/sample_submission/agent.yaml')
if path.exists():
    content = path.read_text(encoding='utf-8')
    print("=== agent.yaml ===")
    print(content)
    try:
        parsed = yaml.safe_load(content)
        print("\n[OK] Файл agent.yaml валиден как YAML")
    except Exception as e:
        print(f"\n[ERROR] Ошибка синтаксиса YAML: {e}")
else:
    print("[ERROR] Файл agent.yaml не найден")

=== agent.yaml ===
agent_class: GemmaDeveloperAgent
model: gemma-4-31b-it-qat-w4a16-ct
max_turns: 8
prompt_path: prompts/system.md
sub_agents:
  - sub_agents/code_analyzer.yaml

[OK] Файл agent.yaml валиден как YAML
